<a href="https://colab.research.google.com/github/SouravDasz/Deep-learning-projects-models-only-/blob/main/image_caption_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! mkdir -p ~/.kaggle

In [ ]:
! cp kaggle.json ~/.kaggle/

In [ ]:
! chmod 600 ~/.kaggle/kaggle.json

In [4]:
! kaggle datasets download adityajn105/flickr8k

Dataset URL: https://www.kaggle.com/datasets/adityajn105/flickr8k
License(s): CC0-1.0
100% 1.04G/1.04G [01:10<00:00, 15.9MB/s]



In [5]:
import zipfile
zip=zipfile.ZipFile("/content/flickr8k.zip")
zip.extractall("/content")
zip.close()

In [6]:
lis=[]
with open("/content/captions.txt","r") as r:
  next(r)
  for line in r:
    line=line.strip().split(",",1)
    lis.append(line)

In [7]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset , DataLoader
import torch.optim as optim
import PIL
import cv2

In [8]:
lis[2][0]

'1000268201_693b08cb0e.jpg'

In [9]:
vocab={
    "<PAD>":0,
    "<SOS>":1,
    "<EOS>":2,
    "<UNK>":3
}

In [10]:
for img,cap in lis:
  cap=cap.split()
  cap=cap[:-1]

  for word in cap:
    if word not in vocab:
      vocab[word]=len(vocab)

In [11]:
vocab

{'<PAD>': 0,
 '<SOS>': 1,
 '<EOS>': 2,
 '<UNK>': 3,
 'A': 4,
 'child': 5,
 'in': 6,
 'a': 7,
 'pink': 8,
 'dress': 9,
 'is': 10,
 'climbing': 11,
 'up': 12,
 'set': 13,
 'of': 14,
 'stairs': 15,
 'an': 16,
 'entry': 17,
 'way': 18,
 'girl': 19,
 'going': 20,
 'into': 21,
 'wooden': 22,
 'building': 23,
 'little': 24,
 'playhouse': 25,
 'the': 26,
 'to': 27,
 'her': 28,
 'cabin': 29,
 'black': 30,
 'dog': 31,
 'and': 32,
 'spotted': 33,
 'are': 34,
 'tri-colored': 35,
 'playing': 36,
 'with': 37,
 'each': 38,
 'other': 39,
 'on': 40,
 'road': 41,
 'white': 42,
 'brown': 43,
 'spots': 44,
 'staring': 45,
 'at': 46,
 'street': 47,
 'Two': 48,
 'dogs': 49,
 'different': 50,
 'breeds': 51,
 'looking': 52,
 'pavement': 53,
 'moving': 54,
 'toward': 55,
 'covered': 56,
 'paint': 57,
 'sits': 58,
 'front': 59,
 'painted': 60,
 'rainbow': 61,
 'hands': 62,
 'bowl': 63,
 'sitting': 64,
 'large': 65,
 'small': 66,
 'grass': 67,
 'plays': 68,
 'fingerpaints': 69,
 'canvas': 70,
 'it': 71,
 'There'

In [12]:
len_list=[]
for _,cap in lis:
  cap=cap.split()
  cap=cap[:-1]
  cap.insert(0,"<SOS>")
  cap.append("<EOS>")
  len_list.append(len(cap))

In [13]:
max(len_list)

39

In [14]:
def integar_encoding(cap,vocab):
  # Split the caption into words and remove the last element (e.g., period)
  words = cap.split()
  words = words[:-1]

  # Initialize the encoded list with <SOS> token
  encoded_caption = [vocab["<SOS>"]]

  # Encode each word, using <UNK> for unknown words
  for word in words:
    encoded_caption.append(vocab.get(word, vocab["<UNK>"]))

  # Add the <EOS> token at the end
  encoded_caption.append(vocab["<EOS>"])
  if len(encoded_caption) < 39:
    encoded_caption = encoded_caption + [vocab["<PAD>"]] * (39 - len(encoded_caption))
  else:
    encoded_caption = encoded_caption[:39]
  return encoded_caption

In [18]:
device= "cuda" if torch.cuda.is_available() else "cpu"

In [19]:
# Dataset
import os
class Custom_Dataset(Dataset):
  def __init__(self,lis,vocab,path):
    self.lis=lis
    self.vocab=vocab
    self.path=path
  def __len__(self):
    return len(self.lis)
  def __getitem__(self,idx):
    img_filename = self.lis[idx][0]
    caption_text = self.lis[idx][1]

    # Construct the full image path
    full_img_path = os.path.join(self.path, img_filename)

    # Read the image directly using cv2.imread
    img = cv2.imread(full_img_path)

    # Check if image was loaded successfully
    if img is None:
        raise FileNotFoundError(f"Image not found at {full_img_path}")

    img=cv2.resize(img,(224,224))
    img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    img = torch.tensor(img, dtype=torch.float32) / 255.0
    img = img.permute(2, 0, 1)
    cap = integar_encoding(caption_text, self.vocab)
    cap=torch.tensor(cap,dtype=torch.long)
    img=img.to(device)
    cap=cap.to(device)
    return img,cap

In [20]:
data=Custom_Dataset(lis,vocab,"/content/Images/")

In [21]:
data[2]

(tensor([[[0.2980, 0.4157, 0.4902,  ..., 0.0157, 0.0196, 0.0235],
          [0.2863, 0.4314, 0.5020,  ..., 0.0353, 0.0275, 0.0549],
          [0.2745, 0.4471, 0.4980,  ..., 0.0275, 0.0118, 0.0118],
          ...,
          [0.8353, 0.5608, 0.6627,  ..., 0.8745, 0.6431, 0.6431],
          [0.6510, 0.6235, 0.8863,  ..., 0.8510, 0.6745, 0.6588],
          [0.8039, 0.8627, 0.7020,  ..., 0.8196, 0.6667, 0.6627]],
 
         [[0.3176, 0.4824, 0.5647,  ..., 0.0078, 0.0196, 0.0353],
          [0.2980, 0.4941, 0.5804,  ..., 0.0392, 0.0314, 0.0784],
          [0.2784, 0.5098, 0.5804,  ..., 0.0353, 0.0392, 0.0627],
          ...,
          [0.5725, 0.2549, 0.4745,  ..., 0.8941, 0.7255, 0.7255],
          [0.3529, 0.4353, 0.6588,  ..., 0.8667, 0.7255, 0.7216],
          [0.4902, 0.5686, 0.4039,  ..., 0.8314, 0.7294, 0.7216]],
 
         [[0.3608, 0.4588, 0.5412,  ..., 0.0118, 0.0157, 0.0196],
          [0.3373, 0.4863, 0.5608,  ..., 0.0314, 0.0353, 0.0353],
          [0.3412, 0.5098, 0.5686,  ...,

In [22]:
data_loader=DataLoader(data,batch_size=64,shuffle=True)

In [23]:
import  torchvision.models as models

In [24]:
models.resnet50(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 146MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [25]:
# encoder

class Encoder (nn.Module):
  def __init__(self,embed_size):
    super().__init__()
    self.embed_size=embed_size
    resnet=models.resnet50(pretrained=True)
    # Remove the final classification layer by taking all children except the last one
    self.resnet_features = nn.Sequential(*list(resnet.children())[:-1])
    self.linear=nn.Linear(resnet.fc.in_features, embed_size)
    self.relu=nn.ReLU()
  def forward(self,x):
    for param in self.resnet_features.parameters():
      param.requires_grad = False
    x=self.resnet_features(x) # Output will be (batch_size, 2048, 1, 1) from avgpool
    x=x.view(x.size(0), -1) # Flatten to (batch_size, 2048)
    x=self.linear(x)
    x=self.relu(x)
    return x

In [26]:
# Decoder

class Decoder(nn.Module):
  def __init__(self,embed_size,hidden_size):
    super().__init__()
    self.embed_size=embed_size
    self.hidden_size=hidden_size
    self.embedding=nn.Embedding(len(vocab),self.embed_size)
    self.lstm=nn.LSTM(self.embed_size,hidden_size=hidden_size,batch_first=True)
    self.linear=nn.Linear(self.hidden_size,len(vocab))
  def forward(self,feature,caption):
    caption=caption[:,:-1]
    emb=self.embedding(caption)
    emb=torch.cat((feature.unsqueeze(1),emb),dim=1)
    hidden,_=self.lstm(emb)
    output=self.linear(hidden)
    return output

In [27]:
class Caption_generator(nn.Module):
  def __init__(self,encoder,decoder):
    super().__init__()
    self.encoder=encoder
    self.decoder=decoder
  def forward(self,x,caption):
    feature=self.encoder(x)
    output=self.decoder(feature,caption)
    return output

In [28]:
emb_size=256
hidden_size=512
encoder=Encoder(emb_size)
decoder=Decoder(emb_size,hidden_size)

In [30]:
model=Caption_generator(encoder,decoder)

In [31]:
critrion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

In [32]:
model.to(device)

Caption_generator(
  (encoder): Encoder(
    (resnet_features): Sequential(
      (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (4): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplac

In [33]:

for epochs in range(15):
  epocha_loss=0
  for img,cap in  data_loader:

    optimizer.zero_grad()
    output=model(img,cap)
    output=output.reshape(-1,output.shape[2])
    cap=cap.reshape(-1)
    loss=critrion(output,cap)
    loss.backward()
    optimizer.step()
    epocha_loss+=loss.item()
  print(f"epoch:{epochs} loss:{epocha_loss/len(data_loader)}")

epoch:0 loss:1.284313300962885
epoch:1 loss:0.977717438203652
epoch:2 loss:0.8747188601448638
epoch:3 loss:0.7991210049927517
epoch:4 loss:0.7366600468825389
epoch:5 loss:0.6806897745124914
epoch:6 loss:0.6297723428709624
epoch:7 loss:0.5837328739075864
epoch:8 loss:0.540604385625318
epoch:9 loss:0.5015570060901732
epoch:10 loss:0.4655391686900532
epoch:11 loss:0.4341437742702581
epoch:12 loss:0.40490543089018427
epoch:13 loss:0.3773825449212844
epoch:14 loss:0.35332103913994195


In [ ]:
def prediction(model,img,vocab):
  model.eval()
  result_list=[]
  with torch.no_grad():
    feature=model.encoder(img)
    feature=feature.unsqueeze(1)
    state=None
    for _ in range(39):
      output,state=model.decoder.lstm(feature,state)
      output=model.decoder.linear(output.squeeze(1))
      pred=output.argmax(1)
      word = vocab.itos[predicted.item()]
      result_caption.append(word)

      # Stop if <end>
      if word == "<end>":
          break

      # Step 4: Next input = embedding of predicted word
      inputs = model.decoder.embedding(predicted)
      inputs = inputs.unsqueeze(1)   # (1, 1, embed_size)

    return result_caption